# Lab 3: Conditional Edges (Applied Version)
In this notebook, we'll evolve our **Smart Content Generation System** by adding a feedback loop.

After the parallel SEO and Readability checks, we want an **Evaluation Node** to inspect the feedback. If the quality is too low or contains errors, the graph should route *back* to the draft node for a rewrite. If it's satisfactory, it goes to completion.

We use the following concepts:
- **Conditional Routing**: Deciding dynamically between rewriting the draft or ending the graph based on state variables.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Verify API keys
print("OpenAI API Key set:", "OPENAI_API_KEY" in os.environ)
print("LangSmith tracing set:", os.environ.get("LANGCHAIN_TRACING_V2"))

OpenAI API Key set: True
LangSmith tracing set: true


### 1. Define State and Nodes
We track `rewrites_count` and `approved` flag to prevent infinite loops.

In [2]:
from typing import TypedDict, List, Annotated
import operator
import json
from mock_llm import get_llm
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END

class ContentState(TypedDict):
    topic: str
    draft: str
    feedback_list: Annotated[List[str], operator.add]
    rewrites_count: int
    approved: bool
    status: str

llm = get_llm(model="gpt-4o-mini", temperature=0.5)

def generate_draft_node(state: ContentState):
    print(f"--- Node: Generating Draft (Rewrite Count: {state.get('rewrites_count', 0)}) ---")
    # Build prompt, incorporating previous feedback list if it exists
    feedback_str = "\n- ".join(state.get("feedback_list", []))
    instructions = "Write a brief paragraph (under 100 words) about the topic."
    if feedback_str:
        instructions += f"\nPlease revise the previous draft to address this feedback:\n- {feedback_str}"
        
    prompt = ChatPromptTemplate.from_messages([
        ("system", instructions),
        ("human", "Topic: {topic}")
    ])
    response = (prompt | llm).invoke({"topic": state["topic"]})
    return {
        "draft": response.content.strip(),
        "status": "drafted",
        # Clear the old feedback list for the next round of analysis
        "feedback_list": [] 
    }

def seo_audit_node(state: ContentState):
    print("--- Node: Conducting SEO Audit ---")
    # Simple heuristic check for demo: must contain keyword 'future'
    if "future" not in state["draft"].lower():
        return {"feedback_list": ["SEO: The article must include the keyword 'future'."]}
    return {"feedback_list": []}

def readability_check_node(state: ContentState):
    print("--- Node: Checking Readability ---")
    # Heuristic: must be under 80 words for readability
    word_count = len(state["draft"].split())
    if word_count > 80:
        return {"feedback_list": [f"Readability: Draft is too long ({word_count} words). Shorten it to under 80 words."]}
    return {"feedback_list": []}

# Decision Node: Evaluate quality
def evaluate_quality_node(state: ContentState):
    print("--- Node: Evaluating Quality ---")
    feedback = state["feedback_list"]
    # Approve if no feedback is generated, or if we have rewritten twice already
    should_approve = len(feedback) == 0 or state.get("rewrites_count", 0) >= 2
    
    return {
        "approved": should_approve,
        "rewrites_count": state.get("rewrites_count", 0) + (0 if should_approve else 1),
        "status": "approved" if should_approve else "rejected"
    }

# Routing function for conditional edge
def route_after_evaluation(state: ContentState):
    if state["approved"]:
        print("-> Route: Graph Approved!")
        return "approved_path"
    else:
        print(f"-> Route: Needs revision. Feedback list count: {len(state['feedback_list'])}")
        return "rewrite_path"

--- OpenAI API connection failed (Error code: 429 - {'error': {'message': 'You exceeded your c...). Falling back to Mock LLM ---


### 2. Build and Compile Graph
We set up conditional logic after the `evaluate` node: loops back to `generate_draft` on reject, goes to `END` on approval.

In [3]:
builder = StateGraph(ContentState)
builder.add_node("generate_draft", generate_draft_node)
builder.add_node("seo_audit", seo_audit_node)
builder.add_node("readability_check", readability_check_node)
builder.add_node("evaluate", evaluate_quality_node)

builder.add_edge(START, "generate_draft")
builder.add_edge("generate_draft", "seo_audit")
builder.add_edge("generate_draft", "readability_check")
builder.add_edge("seo_audit", "evaluate")
builder.add_edge("readability_check", "evaluate")

# Configure Conditional Edge from evaluate node
builder.add_conditional_edges(
    "evaluate",
    route_after_evaluation,
    {
        "approved_path": END,
        "rewrite_path": "generate_draft"
    }
)

graph = builder.compile()

### 3. Run and Observe Loop
We pass a topic. The graph will run, evaluate rules (e.g. check for word counts and keyword 'future'), generate feedback, loop back if necessary, and finalize.

In [4]:
initial_input = {
    "topic": "What AI tools can do for writing copy",
    "draft": "",
    "feedback_list": [],
    "rewrites_count": 0,
    "approved": False,
    "status": "pending"
}

final_state = graph.invoke(initial_input)
print("\n--- Final Summary ---")
print("Rewrites Count:", final_state["rewrites_count"])
print("Final status:", final_state["status"])
print("Final Text:\n", final_state["draft"])

--- Node: Generating Draft (Rewrite Count: 0) ---
--- Node: Checking Readability ---
--- Node: Conducting SEO Audit ---
--- Node: Evaluating Quality ---
-> Route: Needs revision. Feedback list count: 1
--- Node: Generating Draft (Rewrite Count: 1) ---
--- Node: Checking Readability ---
--- Node: Conducting SEO Audit ---
--- Node: Evaluating Quality ---
-> Route: Needs revision. Feedback list count: 1
--- Node: Generating Draft (Rewrite Count: 2) ---
--- Node: Checking Readability ---
--- Node: Conducting SEO Audit ---
--- Node: Evaluating Quality ---
-> Route: Graph Approved!

--- Final Summary ---
Rewrites Count: 2
Final status: approved
Final Text:
 The future of AI tools in copy writing is bright. AI drafts copy instantly, analyzing brand voices and generating ideas. Copywriters use AI to beat writer's block and shape the future of digital marketing.
